Akarsh Dubey        ( CS25MTECH14001 )             
Atish Kadam         ( CS25MTECH14003 )      
Debdip Choudhuri    ( CS25MTECH11025 )

In [2]:

import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
import pandas as pd
import time
import torchvision.models as models
import numpy as np
from sklearn.model_selection import StratifiedKFold

# =============================================
# Dataset
# =============================================
class HackathonDataset(Dataset):
    def __init__(self, data_dir, split='train', transform=None):
        self.data_dir = data_dir
        self.split = split
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.image_ids = []
        
        if split == 'train':
            class_0_dir = os.path.join(data_dir, 'train', 'train', '0')
            if not os.path.exists(class_0_dir):
                class_0_dir = os.path.join(data_dir, 'train', '0')
            if os.path.exists(class_0_dir):
                paths_0 = glob.glob(os.path.join(class_0_dir, '*.*'))
                self.image_paths.extend(paths_0)
                self.labels.extend([0] * len(paths_0))
            
            class_1_dir = os.path.join(data_dir, 'train', 'train', '1')
            if not os.path.exists(class_1_dir):
                class_1_dir = os.path.join(data_dir, 'train', '1')
            if os.path.exists(class_1_dir):
                paths_1 = glob.glob(os.path.join(class_1_dir, '*.*'))
                self.image_paths.extend(paths_1)
                self.labels.extend([1] * len(paths_1))
        elif split == 'test':
            test_dir = os.path.join(data_dir, 'test', 'test')
            if not os.path.exists(test_dir):
                test_dir = os.path.join(data_dir, 'test')
            if os.path.exists(test_dir):
                self.image_paths = glob.glob(os.path.join(test_dir, '*.*'))
            self.image_ids = [os.path.basename(p) for p in self.image_paths]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        if self.split == 'train':
            return image, self.labels[idx]
        else:
            return image, self.image_ids[idx]

# =============================================
# Global Sum Pooling — acts as object counter
# =============================================
class GlobalSumPool2d(nn.Module):
    def forward(self, x):
        return x.sum(dim=[2, 3])

# =============================================
# Modified ResNet-18: SumPool + MLP head
# =============================================
class ModifiedResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights=None)
        self.features = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4
        )
        self.pool = GlobalSumPool2d()
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

# =============================================
# Main Pipeline: 4-Fold Ensemble + Early Stop
# =============================================
def generate_predictions(data_dir):
    print(f"Starting pipeline with data directory: {data_dir}")
    
    # Hyperparameters
    batch_size = 64
    max_epochs = 40
    learning_rate = 1e-3
    weight_decay = 1e-4
    patience = 7
    n_folds = 4
    img_size = 224
    
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")
    
    train_transforms = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),  # ADDED
        transforms.RandomHorizontalFlip(),                          # ADDED
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transforms = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print("Loading datasets...")
    full_train_aug = HackathonDataset(data_dir, split='train', transform=train_transforms)
    full_train_clean = HackathonDataset(data_dir, split='train', transform=val_transforms)
    test_dataset = HackathonDataset(data_dir, split='test', transform=val_transforms)
    
    if len(full_train_aug) == 0:
        print("Warning: Train dataset is empty. Check data_dir structure.")
        return
    
    print(f"Train images: {len(full_train_aug)} | Test images: {len(test_dataset)}")
    
    targets = np.array(full_train_aug.labels)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
    fold_model_paths = []
    fold_val_accs = []
    
    total_start = time.time()
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(targets)), targets)):
        print(f"\n{'='*50}")
        print(f"  FOLD {fold+1}/{n_folds}")
        print(f"{'='*50}")
        
        train_sub = Subset(full_train_aug, train_idx)
        val_sub = Subset(full_train_clean, val_idx)
        
        train_loader = DataLoader(train_sub, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
        val_loader = DataLoader(val_sub, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
        
        model = ModifiedResNet18().to(device)
        criterion = nn.BCEWithLogitsLoss()  # NO pos_weight needed — classes are balanced (9001 each)
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
        
        best_val_acc = 0.0
        epochs_no_improve = 0
        best_path = f'best_model_fold_{fold}.pth'
        
        fold_start = time.time()
        
        for epoch in range(max_epochs):
            # ---- Train ----
            model.train()
            running_loss = 0.0
            correct = 0
            total = 0
            
            for inputs, labels in train_loader:
                inputs = inputs.to(device)
                labels = labels.float().to(device)
                optimizer.zero_grad()
                
                if scaler:
                    with torch.amp.autocast('cuda'):
                        outputs = model(inputs).squeeze(1)
                        loss = criterion(outputs, labels)
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    outputs = model(inputs).squeeze(1)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                predicted = (torch.sigmoid(outputs) >= 0.5).long()
                total += labels.size(0)
                correct += predicted.eq(labels.long()).sum().item()
            
            scheduler.step()
            train_acc = 100. * correct / total
            
            # ---- Validate ----
            model.eval()
            val_correct = 0
            val_total = 0
            
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs = inputs.to(device)
                    labels = labels.float().to(device)
                    if scaler:
                        with torch.amp.autocast('cuda'):
                            outputs = model(inputs).squeeze(1)
                    else:
                        outputs = model(inputs).squeeze(1)
                    predicted = (torch.sigmoid(outputs) >= 0.5).long()
                    val_total += labels.size(0)
                    val_correct += predicted.eq(labels.long()).sum().item()
            
            val_acc = 100. * val_correct / val_total
            print(f"  Epoch {epoch+1:02d}/{max_epochs} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                epochs_no_improve = 0
                torch.save(model.state_dict(), best_path)
                print(f"    -> Saved best model (Val Acc: {val_acc:.2f}%)")
            else:
                epochs_no_improve += 1
            
            if epochs_no_improve >= patience:
                print(f"    -> Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
                break
        
        fold_model_paths.append(best_path)
        fold_val_accs.append(best_val_acc)
        print(f"  Fold {fold+1} done in {(time.time()-fold_start)/60:.1f} min | Best Val Acc: {best_val_acc:.2f}%")
    
    print(f"\n{'='*50}")
    print(f"  ALL FOLDS COMPLETE")
    print(f"{'='*50}")
    print(f"Total training time: {(time.time()-total_start)/60:.1f} minutes")
    for i, acc in enumerate(fold_val_accs):
        print(f"  Fold {i+1}: {acc:.2f}%")
    print(f"  Mean Val Acc: {np.mean(fold_val_accs):.2f}%")
    
    # ==========================================
    # Ensemble Inference (soft voting)
    # ==========================================
    print(f"\nGenerating ensembled predictions on test set...")
    
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
    
    all_probs_sum = None
    all_ids = []
    
    if len(test_dataset) > 0:
        for i, model_path in enumerate(fold_model_paths):
            print(f"  Inference with Fold {i+1} model...")
            model = ModifiedResNet18().to(device)
            model.load_state_dict(torch.load(model_path, map_location=device))
            model.eval()
            
            fold_probs = []
            fold_ids = []
            
            with torch.no_grad():
                for inputs, image_ids in test_loader:
                    inputs = inputs.to(device)
                    if device.type == 'cuda':
                        with torch.amp.autocast('cuda'):
                            outputs = model(inputs).squeeze(1)
                    else:
                        outputs = model(inputs).squeeze(1)
                    
                    probs = torch.sigmoid(outputs)
                    fold_probs.extend(probs.cpu().numpy())
                    if i == 0:
                        fold_ids.extend(image_ids)
            
            fold_probs = np.array(fold_probs)
            if all_probs_sum is None:
                all_probs_sum = fold_probs
                all_ids = fold_ids
            else:
                all_probs_sum += fold_probs
        
        avg_probs = all_probs_sum / n_folds
        final_preds = (avg_probs >= 0.5).astype(int)  # 0.5 threshold — classes are balanced
        
        submission_df = pd.DataFrame({
            'ID': all_ids,
            'Label': final_preds
        })
        
        output_path = 'submission.csv'
        submission_df.to_csv(output_path, index=False)
        print(f"\nSUCCESS: Ensembled predictions saved to {output_path}")
    else:
        print("Test dataset is empty, skipping predictions.")

if __name__ == "__main__":
    data_dir = "/kaggle/input/competitions/iith-deep-learning-2026-hackathon"
    generate_predictions(data_dir)


Starting pipeline with data directory: /kaggle/input/competitions/iith-deep-learning-2026-hackathon
Using device: cuda
Loading datasets...
Train images: 18000 | Test images: 5010

  FOLD 1/4
  Epoch 01/40 | Train Acc: 74.84% | Val Acc: 95.96%
    -> Saved best model (Val Acc: 95.96%)
  Epoch 02/40 | Train Acc: 89.60% | Val Acc: 96.16%
    -> Saved best model (Val Acc: 96.16%)
  Epoch 03/40 | Train Acc: 91.73% | Val Acc: 98.27%
    -> Saved best model (Val Acc: 98.27%)
  Epoch 04/40 | Train Acc: 92.81% | Val Acc: 97.00%
  Epoch 05/40 | Train Acc: 93.83% | Val Acc: 99.07%
    -> Saved best model (Val Acc: 99.07%)
  Epoch 06/40 | Train Acc: 93.70% | Val Acc: 98.89%
  Epoch 07/40 | Train Acc: 94.85% | Val Acc: 98.13%
  Epoch 08/40 | Train Acc: 95.16% | Val Acc: 97.13%
  Epoch 09/40 | Train Acc: 96.01% | Val Acc: 98.56%
  Epoch 10/40 | Train Acc: 95.89% | Val Acc: 99.24%
    -> Saved best model (Val Acc: 99.24%)
  Epoch 11/40 | Train Acc: 96.78% | Val Acc: 98.04%
  Epoch 12/40 | Train Acc: 